# Try DataRanger

Watch a patrol find planted problems in demo data, fix what it can verify, and leave the rest for review. No database credentials, API key or GPU is needed.

In Colab, choose **Runtime → Run all** with a standard CPU runtime. The first cell installs the released package; the remaining cells use that runtime.

## Install DataRanger

This example uses the released **0.3.0** package from PyPI. Installation time depends on the runtime and network.

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "dataranger==0.3.0"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(result.stdout, end="")
result.check_returncode()

## Run a patrol

The demo generates 60 days of synthetic transactions with known problems. The summary reports how many incidents were fixed, left unchanged or escalated for a person. Counts can vary with the date because the generated days follow the current calendar.

Each run writes to a new folder, so rerunning this cell keeps earlier reports.

In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

report_dir = Path(tempfile.mkdtemp(prefix="dataranger-demo-", dir=".")).resolve()
result = subprocess.run(
    [sys.executable, "-m", "dataranger.cli", "demo", "--seed", "7", "--out", str(report_dir)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(result.stdout, end="")
result.check_returncode()

## Read the audit log

Each line in `audit.jsonl` describes an incident and its evidence. These first five rows show the affected partition, detected problems, decision and chosen candidate. A chosen candidate may still be escalated; check the decision to see whether it was applied.

In [ ]:
import json
import pandas as pd

audit_path = report_dir / "audit.jsonl"
records = [json.loads(line) for line in audit_path.read_text(encoding="utf-8").splitlines()]
audit_preview = pd.DataFrame([
    {
        "Partition": record.get("partition") or "Whole table",
        "Problems": ", ".join(record.get("detectors") or ["schema_contract"]),
        "Decision": record["decision"],
        "Chosen candidate": (record.get("chosen") or {}).get("fix", "None"),
    }
    for record in records
]).head(5)
audit_preview

## Explore the dashboard

Select an incident to see the evidence, or switch between **Needs you**, **Fixed** and **Left alone**. **Audit log** downloads the original JSONL file. The embedded page keeps its styles separate from the notebook.

The approval and rollback buttons only change this page's in-memory display; they do not rewrite the saved audit or modify a database. Rerunning this cell resets those display choices.

In [ ]:
import base64
import html
import warnings
from IPython.display import HTML

dashboard_html = (report_dir / "dashboard.html").read_text(encoding="utf-8")
audit_data = base64.b64encode(audit_path.read_bytes()).decode("ascii")
# Adapt links and action notes in the embedded copy; leave the saved dashboard unchanged.
dashboard_html = dashboard_html.replace("<head>", '<head><base href="about:srcdoc">', 1)
dashboard_html = dashboard_html.replace(
    'href="audit.jsonl"',
    f'href="data:application/x-ndjson;base64,{audit_data}" download="audit.jsonl"',
)
dashboard_html = dashboard_html.replace(
    "Your decision is recorded in the audit log and used the next time the gate is calibrated.",
    "Choices in this demo affect this view only; the saved audit stays unchanged.",
).replace(
    "It will be used to recalibrate the gate.",
    "This preview does not save decisions.",
)
# HTML with srcdoc is intentional here; ignore only IPython's class-choice advisory.
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore", message=r"^Consider using IPython\.display\.IFrame instead$",
        category=UserWarning, module=r"^IPython\.core\.display$",
    )
    dashboard_view = HTML(
        '<iframe title="DataRanger patrol dashboard" width="100%" height="850" '
        'style="border: 0" sandbox="allow-scripts allow-downloads" '
        f'srcdoc="{html.escape(dashboard_html, quote=True)}"></iframe>'
    )
dashboard_view

## Keep the report or try your tables

In Colab's **Files** sidebar, open the folder printed by the demo to download `dashboard.html`, `audit.jsonl`, `incidents.csv` or `fixed_version.csv`. Keep the dashboard and audit together when opening the report outside the notebook. Runtime files are temporary; download anything you want to keep before disconnecting.

The demo uses synthetic data and a benchmark-calibrated threshold. For your own tables and calibration, see the [DataRanger README](https://github.com/akshaysharma1088/dataranger#try-it).